In [8]:
import wandb
import torch
from transformers import (
    AutoConfig,
    AutoModel,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    GenerationConfig,
    PretrainedConfig,
    PreTrainedModel,
    get_scheduler,
    AutoTokenizer
)
from peft import get_peft_model, LoraConfig, PeftModel
from datasets import load_dataset
import json
from tqdm import trange
from rlhf import first_true_indices

eval_set = load_dataset("tatsu-lab/alpaca_eval", "alpaca_eval")["eval"]
def truncate_response(truncate_token_id: int, tokenizer: AutoTokenizer, responses: torch.Tensor) -> torch.Tensor:
    """
    Truncates responses after the first occurrence of the truncate token.
    """
    trunc_idxs = first_true_indices(responses == truncate_token_id).unsqueeze(-1)
    new_size = [1] * (len(responses.size()) - 1) + [responses.shape[1]]
    idxs = torch.arange(responses.shape[1], device=responses.device).view(*new_size)
    postprocessed_responses = torch.masked_fill(responses, idxs > trunc_idxs, tokenizer.pad_token_id)
    return postprocessed_responses

def generate_responses(lora_dir: str):
    base_model = AutoModelForCausalLM.from_pretrained(
        "meta-llama/Llama-3.1-8B-Instruct",
        torch_dtype="auto",
        device_map="auto"
    )

    config = LoraConfig.from_pretrained(lora_dir)
    policy = PeftModel.from_pretrained(base_model, lora_dir, config = config, torch_dtype="auto", device_map="auto")

    policy.generation_config.eos_token_id = None  # disable `pad_token_id` and `eos_token_id` because we want to generate tokens without truncation / padding
    policy.generation_config.pad_token_id = None 

    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
    tokenizer.padding_side = "left"
    pairs = [(tokenizer.apply_chat_template([{"role": "user", "content": sample["instruction"]}],
                                            add_generation_prompt=True,
                                            padding = "max_length",
                                            max_length = 256 + 64,
                                            truncation = True,
                                            return_tensors="pt"), sample["instruction"]) for sample in eval_set]

    prompt_token_ids, instructions = zip(*pairs)
    prompt_token_ids = torch.cat(prompt_token_ids).to(policy.device)
    print(prompt_token_ids.shape)
    policy.eval()

    MINIBATCH_SIZE = 256
    all_responses = []
    for i in trange(0, len(prompt_token_ids), MINIBATCH_SIZE):   
        batch_token_ids = prompt_token_ids[i:i+MINIBATCH_SIZE].to(policy.device)
        attention_mask = batch_token_ids != tokenizer.pad_token_id
        input_ids = torch.masked_fill(batch_token_ids, ~attention_mask, 0)
        responses = policy.generate(inputs = input_ids,
                                attention_mask = attention_mask,
                                max_new_tokens = 1024,
                                min_new_tokens = 1024)
        responses = responses[:, 256 + 64:]
        responses = truncate_response(tokenizer.eos_token_id, tokenizer, responses)
        all_responses.extend(responses)
    
    json_outputs = []
    for instruction, response in zip(instructions, all_responses):
        json_outputs.append({"instruction": instruction, 
                             "output": tokenizer.decode(response, skip_special_tokens = True),
                             "generator": lora_dir})
        
    for element in json_outputs:
        element["output"] = element["output"][element["output"].find("assistant")+10:]

    return json_outputs

def generate_baseline_responses():
    policy = AutoModelForCausalLM.from_pretrained(
        "meta-llama/Llama-3.1-8B-Instruct",
        torch_dtype="auto",
        device_map="auto"
    )
    policy.generation_config.eos_token_id = None  # disable `pad_token_id` and `eos_token_id` because we want to generate tokens without truncation / padding
    policy.generation_config.pad_token_id = None 

    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
    tokenizer.padding_side = "left"
    pairs = [(tokenizer.apply_chat_template([{"role": "user", "content": sample["instruction"]}],
                                            add_generation_prompt=True,
                                            padding = "max_length",
                                            max_length = 256 + 64,
                                            truncation = True,
                                            return_tensors="pt"), sample["instruction"]) for sample in eval_set]

    prompt_token_ids, instructions = zip(*pairs)
    prompt_token_ids = torch.cat(prompt_token_ids).to(policy.device)
    print(prompt_token_ids.shape)
    policy.eval()

    MINIBATCH_SIZE = 256
    all_responses = []
    for i in trange(0, len(prompt_token_ids), MINIBATCH_SIZE):   
        batch_token_ids = prompt_token_ids[i:i+MINIBATCH_SIZE].to(policy.device)
        attention_mask = batch_token_ids != tokenizer.pad_token_id
        input_ids = torch.masked_fill(batch_token_ids, ~attention_mask, 0)
        responses = policy.generate(inputs = input_ids,
                                attention_mask = attention_mask,
                                max_new_tokens = 1024,
                                min_new_tokens = 1024)
        responses = responses[:, 256 + 64:]
        responses = truncate_response(tokenizer.eos_token_id, tokenizer, responses)
        all_responses.extend(responses)
    
    json_outputs = []
    for instruction, response in zip(instructions, all_responses):
        json_outputs.append({"instruction": instruction, 
                             "output": tokenizer.decode(response, skip_special_tokens = True),
                             "generator": "Baseline"})
        
    for element in json_outputs:
        element["output"] = element["output"][element["output"].find("assistant")+10:]

    return json_outputs

In [2]:
json_outputs_1 = generate_responses("ultrafeedback_rloo")
import json
with open("alpaca_eval_results_rloo.json", "w") as f:
    json.dump(json_outputs_1, f)

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.17it/s]
/home/dev/.local/lib/python3.11/site-packages/peft/utils/save_and_load.py:224: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this

torch.Size([805, 320])


100%|██████████| 4/4 [13:21<00:00, 200.39s/it]


In [ ]:
json_outputs_2 = generate_responses("ultrafeedback_rloo_patch")
import json
with open("alpaca_eval_results_rloo_patch.json", "w") as f:
    json.dump(json_outputs_2, f)

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.21it/s]
/home/dev/.local/lib/python3.11/site-packages/peft/utils/save_and_load.py:224: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this

torch.Size([805, 320])


 25%|██▌       | 1/4 [04:11<12:35, 251.80s/it]

In [9]:
json_outputs_3 = generate_baseline_responses()
import json
with open("alpaca_eval_results_baseline.json", "w") as f:
    json.dump(json_outputs_3, f)

Loading checkpoint shards: 100%|██████████| 4/4 [00:12<00:00,  3.17s/it]


torch.Size([805, 320])


100%|██████████| 4/4 [13:05<00:00, 196.44s/it]


In [14]:
import json
json_outputs = json_outputs_1 + json_outputs_2
with open("alpaca_eval_results.json", "w") as f:
    json.dump(json_outputs, f)

============================

In [ ]:
rloo_patch_outputs = []
for element in json_outputs:
    if element["generator"] == "ultrafeedback_rloo_patch":
        rloo_patch_outputs.append(element)

import json
with open("alpaca_eval_results_rloo_patch.json", "w") as f:
    json.dump(rloo_patch_outputs, f)

In [18]:
rloo_outputs = []
for element in json_outputs:
    if element["generator"] == "ultrafeedback_rloo":
        rloo_outputs.append(element)

import json
with open("alpaca_eval_results_rloo.json", "w") as f:
    json.dump(rloo_outputs, f)

In [1]:
import json
with open("alpaca_eval_results.json", "r") as f:
    json_outputs = json.load(f)

In [11]:
with open("alpaca_eval_results_rloo.json", "r") as f:
    json_outputs_1 = json.load(f)
with open("alpaca_eval_results_rloo_patch.json", "r") as f:
    json_outputs_2 = json.load(f)
with open("alpaca_eval_results_baseline.json", "r") as f:
    json_outputs_3 = json.load(f)
json_outputs = json_outputs_1 + json_outputs_2
with open("alpaca_eval_results.json", "w") as f:
    json.dump(json_outputs, f)


In [14]:
import pandas as pd
df = pd.read_csv("lb.csv", index_col=0)
df

,win_rate,standard_error,mode,avg_length,n_wins,n_wins_base,n_draws,n_total,discrete_win_rate,length_controlled_winrate,lc_standard_error
ultrafeedback_rloo,42.357042,1.434340,verified,1544,335,461,9,805,42.173913,50.276699,0.448540
ultrafeedback_rloo_patch,41.420867,1.431518,verified,1571,331,466,8,805,41.614907,49.397232,0.487015
